# Instrument C — Tier 2 confirmation gate

Eval-only, GPU. Measures the **chain-preserved scramble margin** on `contrast_pilot_v3/runs_C` (v3C) checkpoints against the frozen bar of **0.05**.

Upload `dist/instrument_c_bundle.zip` to `MyDrive/MIPO_NDD/` first, pick a GPU runtime, then run the cell below. It pins the archive extraction, writes the instrument in, **runs its tests before computing anything**, re-checks training provenance per checkpoint, and scores each one clean vs chain-preserved scramble vs full scramble vs edge-ablated ceiling over several seeds. Results land in `MyDrive/MIPO_NDD/instrument_c_results/<run set>/`.

Read `INSTRUMENT_C_DESIGN.md` in the bundle for what the three modes mean and why a null margin with a ~0 ceiling is *uninformative* rather than negative.


In [ ]:
# Instrument C: the Tier 2 confirmation gate. Paste this whole file into ONE Colab GPU cell.
# Eval-only. Nothing trains; nothing is written into v4/ or any existing run folder.
#
# PREREQUISITE (once): upload dist/instrument_c_bundle.zip to Drive at
#   MyDrive/MIPO_NDD/instrument_c_bundle.zip
# The frozen gate is read on v3C = contrast_pilot_v3/runs_C. Other arms can be appended
# for context (each adds ~10-20 min per checkpoint), e.g. "contrast_pilot_v3/runs_D".
RUN_SETS = ["contrast_pilot_v3/runs_C"]
SEEDS = "0,1,2,3,4"            # several corruption patterns; one seed is not a verdict
BUNDLE_ZIP = "instrument_c_bundle.zip"

from google.colab import drive

drive.mount("/content/drive")
import hashlib
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

NDD = Path("/content/drive/MyDrive/MIPO_NDD")
ARCHIVE = NDD / "mipo_ndd_v4_colab_upload.zip"
assert ARCHIVE.exists(), f"missing {ARCHIVE}"
PROJECT = Path("/content") / f"mipo_ndd_v4_validation_{hashlib.sha256(ARCHIVE.read_bytes()).hexdigest()[:12]}"
if not (PROJECT / "mipo/model.py").exists():
    PROJECT.mkdir(exist_ok=True)
    with zipfile.ZipFile(ARCHIVE) as z:
        z.extractall(PROJECT)
assert "branch_probes" in (PROJECT / "mipo/model.py").read_text(), "NOT v3 code"
print("PROJECT:", PROJECT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[test]"],
               check=True, cwd=PROJECT)

# Write the instrument in from the bundle, then run its tests before any number.
BUNDLE_PATH = NDD / BUNDLE_ZIP
assert BUNDLE_PATH.exists(), f"upload {BUNDLE_ZIP} to {NDD} first"
BUNDLE_DIR = Path("/content/instrument_c_bundle/instrument_c")
if BUNDLE_DIR.exists():
    shutil.rmtree(BUNDLE_DIR.parent)
BUNDLE_DIR.parent.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE_PATH) as z:
    bad = z.testzip()
    assert bad is None, f"bundle zip corrupt at {bad}"
    z.extractall(BUNDLE_DIR.parent)
(PROJECT / "tests").mkdir(exist_ok=True)
# instruments.py is eval-only and cannot change what a checkpoint computes, so an
# archive that predates it is topped up rather than rejected. If the archive already
# carries it, the archive's copy wins and any divergence is reported, not silenced.
digest = lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()[:12]
INSTRUMENTS = PROJECT / "mipo/instruments.py"
if not INSTRUMENTS.exists():
    shutil.copy2(BUNDLE_DIR / "mipo_instruments.py", INSTRUMENTS)
    print("mipo/instruments.py written from bundle (archive predates it)")
elif digest(INSTRUMENTS) == digest(BUNDLE_DIR / "mipo_instruments.py"):
    print("mipo/instruments.py already in archive, identical to bundle")
else:
    print("mipo/instruments.py already in archive and DIFFERS from the bundle; "
          "keeping the archive's copy. The tests below decide whether it is usable.")
for src, dst in [("mipo_instrument_c.py", "mipo/instrument_c.py"),
                 ("evaluate_instrument_c.py", "scripts/evaluate_instrument_c.py"),
                 ("test_instrument_c.py", "tests/test_instrument_c.py"),
                 ("test_instruments.py", "tests/test_instruments.py")]:
    assert (BUNDLE_DIR / src).exists(), f"bundle missing {src}; rebuild it"
    shutil.copy2(BUNDLE_DIR / src, PROJECT / dst)
    print(dst, digest(PROJECT / dst))
subprocess.run([sys.executable, "-m", "pytest", "-q",
                "tests/test_instruments.py", "tests/test_instrument_c.py"], check=True, cwd=PROJECT)

# Training inputs, exactly as the pilot used them. Provenance digests are
# re-checked per checkpoint inside the script, so a mismatch stops the run.
CORPUS = PROJECT / "v4/robust/curriculum.csv.gz"
RESOURCES = PROJECT / "v4/resources"
METADATA = PROJECT / "v4/robust/assay_metadata.csv"
DRIVE_CACHE = NDD / "v4_650M_masked"
LOCAL_CACHE = PROJECT / "v4_650M_masked"
for path in [CORPUS, RESOURCES, METADATA, DRIVE_CACHE / "manifest.json"]:
    assert path.exists(), f"missing {path}"
if not (LOCAL_CACHE / "manifest.json").exists():
    shutil.copytree(DRIVE_CACHE, LOCAL_CACHE, dirs_exist_ok=True)
print("local cache ready:", LOCAL_CACHE)

for run_set in RUN_SETS:
    RUNS = NDD / run_set
    ckpts = sorted(RUNS.rglob("best.pt"))
    assert ckpts, f"no best.pt under {RUNS}"
    print()
    print(f"{run_set}: {len(ckpts)} checkpoint(s):", [str(c.parent.relative_to(RUNS)) for c in ckpts])
    OUT = NDD / "instrument_c_results" / run_set.replace("/", "__")
    subprocess.run([sys.executable, str(PROJECT / "scripts/evaluate_instrument_c.py"),
                    "--runs", str(RUNS), "--corpus", str(CORPUS), "--resources", str(RESOURCES),
                    "--cache", str(LOCAL_CACHE), "--metadata", str(METADATA),
                    "--seeds", SEEDS, "--out", str(OUT)], check=True)
    print("wrote", OUT)
    print(open(OUT / "instrument_c_summary.csv").read())
print("DONE: paste each instrument_c_summary.csv back for the Tier 2 gate verdict")
